<a href="https://colab.research.google.com/github/JSJeong-me/MCP/blob/main/3_LangChain_MCP_%EC%B4%88%EB%B3%B4_%EC%9E%91%EA%B0%80_%EC%98%88%EC%A0%9C_http_fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain + MCP 초보 작가 예제 (Colab/Jupyter fileno 오류 수정본)

이 노트북은 `UnsupportedOperation: fileno` 오류를 피하도록 수정한 버전입니다.

## 왜 오류가 났나요?

원래 예제는 LangChain MCP 클라이언트를 **`stdio` transport**로 연결하고 있었습니다.
그런데 Colab/Jupyter에서는 커널의 표준 입출력 객체가 일반 터미널 파일 디스크립터가 아니어서,
`fileno()` 호출이 필요한 일부 stdio/subprocess 경로에서 오류가 날 수 있습니다.

## 수정 방향

- **기존**: `transport="stdio"`
- **수정**: `transport="http"` 로 변경
- MCP 서버는 FastMCP로 **별도 subprocess HTTP 서버**로 실행
- LangChain은 `MultiServerMCPClient` 로 그 HTTP 서버에 연결

LangChain MCP 문서도 `stdio` 는 로컬 subprocess 통신용이고, `http` 는 HTTP 기반 서버 연결용이라고 설명합니다.


## 실행 순서

1. 패키지 설치
2. MCP weather 서버 파일 생성
3. OpenAI API Key 설정
4. HTTP 서버 subprocess 실행
5. MCP client 연결
6. 도구 목록 조회
7. LangChain agent 생성
8. 작가 시나리오 실행
9. 서버 종료


In [1]:
# 1) 설치
!pip -q install -U fastmcp langchain langchain-openai langchain-mcp-adapters requests


In [2]:
from pathlib import Path

server_lines = [
    "from fastmcp import FastMCP",
    "",
    'mcp = FastMCP("Writing Assistant")',
    "",
    "@mcp.tool()",
    "def get_weather(city: str) -> str:",
    "    '''작가가 특정 도시의 날씨를 물어보면 답변해줍니다.'''",
    "    weather_map = {",
    '        "서울": "현재 서울은 벚꽃이 피기 좋은 화창한 날씨입니다.",',
    '        "부산": "현재 부산은 바닷바람이 부드럽게 부는 맑은 날씨입니다.",',
    '        "제주": "현재 제주는 옅은 안개와 함께 감성적인 새벽 분위기입니다.",',
    "    }",
    '    return weather_map.get(city, f"{city}의 날씨는 소설 속 배경으로 쓰기 좋은 분위기입니다.")',
    "",
    'if __name__ == "__main__":',
    '    mcp.run(transport="http", host="127.0.0.1", port=8000, path="/mcp")',
    "",
]

server_code = "\n".join(server_lines)

server_path = Path("weather_mcp_http.py")
server_path.write_text(server_code, encoding="utf-8")

print(server_path.read_text(encoding="utf-8"))

from fastmcp import FastMCP

mcp = FastMCP("Writing Assistant")

@mcp.tool()
def get_weather(city: str) -> str:
    '''작가가 특정 도시의 날씨를 물어보면 답변해줍니다.'''
    weather_map = {
        "서울": "현재 서울은 벚꽃이 피기 좋은 화창한 날씨입니다.",
        "부산": "현재 부산은 바닷바람이 부드럽게 부는 맑은 날씨입니다.",
        "제주": "현재 제주는 옅은 안개와 함께 감성적인 새벽 분위기입니다.",
    }
    return weather_map.get(city, f"{city}의 날씨는 소설 속 배경으로 쓰기 좋은 분위기입니다.")

if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8000, path="/mcp")



In [3]:
# 3) OpenAI API Key 설정
import os
import getpass

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("Colab Secrets에서 OPENAI_API_KEY를 불러왔습니다.")
    else:
        raise ValueError("Colab Secrets에 OPENAI_API_KEY가 없습니다.")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key 입력: ")
    print("직접 입력 또는 기존 환경변수의 OPENAI_API_KEY를 사용합니다.")


Colab Secrets에서 OPENAI_API_KEY를 불러왔습니다.


## 왜 HTTP transport로 바꿨나요?

LangChain 문서는 MCP 연결에 `stdio` 와 `http` 둘 다 지원한다고 설명합니다.
하지만 Colab/Jupyter에서는 `stdio` subprocess 통신이 커널 입출력과 충돌할 수 있어서,
이 수정본은 **노트북 환경에 더 안전한 HTTP transport** 를 사용합니다.


In [4]:
# 4) HTTP 서버 subprocess 실행
import sys
import time
import socket
import subprocess
import requests

PORT = 8000
MCP_URL = f"http://127.0.0.1:{PORT}/mcp"

server_proc = subprocess.Popen(
    [sys.executable, str(server_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(f"서버 PID: {server_proc.pid}")


서버 PID: 6900


In [5]:
def wait_for_port(host: str, port: int, timeout: float = 20.0):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.2)
    return False

ready = wait_for_port("127.0.0.1", PORT, timeout=20.0)
print("HTTP 서버 준비 상태:", ready)

if not ready:
    if server_proc.poll() is not None and server_proc.stdout is not None:
        print(server_proc.stdout.read())
    raise RuntimeError("HTTP MCP 서버가 제시간에 시작되지 않았습니다.")


HTTP 서버 준비 상태: True


In [6]:
# 선택 확인: HTTP endpoint가 살아있는지 간단히 확인
resp = requests.post(
    MCP_URL,
    json={"jsonrpc": "2.0", "id": 1, "method": "ping"},
    timeout=5
)
print("HTTP endpoint status:", resp.status_code)
print("response text preview:", resp.text[:300])


HTTP endpoint status: 406
response text preview: {"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Not Acceptable: Client must accept both application/json and text/event-stream"}}


In [7]:
# 5) LangChain MCP client 연결 (HTTP transport)
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": MCP_URL,
        }
    }
)

print("MCP client 준비 완료")


MCP client 준비 완료


In [8]:
# 6) 도구 목록 조회
async def show_mcp_tools():
    tools = await client.get_tools()
    print(f"불러온 도구 개수: {len(tools)}")
    for t in tools:
        print({
            "name": getattr(t, "name", None),
            "description": getattr(t, "description", None),
        })
    return tools

tools = await show_mcp_tools()


불러온 도구 개수: 1
{'name': 'get_weather', 'description': '작가가 특정 도시의 날씨를 물어보면 답변해줍니다.'}


In [9]:
# 7) LangChain agent 생성
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4.1",
    tools=tools,
    system_prompt=(
        "당신은 초보 작가를 돕는 창의적인 소설 도우미입니다. "
        "날씨 정보가 필요하면 MCP 도구를 사용하고, "
        "최종 답변은 한국어로 자연스럽고 짧게 작성하세요."
    ),
)

print("LangChain agent 생성 완료")


LangChain agent 생성 완료


In [10]:
# 8) 작가 시나리오 실행
instruction = "지금 서울 날씨를 확인해서, 그 날씨에 어울리는 로맨스 소설의 첫 문장을 한 줄 써줘."

result = await agent.ainvoke(
    {
        "messages": [
            {"role": "user", "content": instruction}
        ]
    }
)

print("\n--- ✨ 작가의 결과물 ---")
for msg in result["messages"]:
    msg_type = getattr(msg, "type", type(msg).__name__)
    content = getattr(msg, "content", "")
    print(f"[{msg_type}] {content}")

print("\n--- ✅ 최종 답변 ---")
print(result["messages"][-1].content)



--- ✨ 작가의 결과물 ---
[human] 지금 서울 날씨를 확인해서, 그 날씨에 어울리는 로맨스 소설의 첫 문장을 한 줄 써줘.
[ai] 
[tool] [{'type': 'text', 'text': '현재 서울은 벚꽃이 피기 좋은 화창한 날씨입니다.', 'id': 'lc_fc3e0854-5fee-45cf-8dbb-ee8a55c91769'}]
[ai] 서울의 화창한 하늘 아래, 벚꽃잎이 흩날리는 거리에서 그와 그녀의 첫 만남이 시작되었다.

--- ✅ 최종 답변 ---
서울의 화창한 하늘 아래, 벚꽃잎이 흩날리는 거리에서 그와 그녀의 첫 만남이 시작되었다.


In [11]:
# 추가 테스트
more_instruction = "지금 부산 날씨를 참고해서, 바닷가를 배경으로 한 소설 첫 문장을 한 줄 써줘."

result2 = await agent.ainvoke(
    {
        "messages": [
            {"role": "user", "content": more_instruction}
        ]
    }
)

print(result2["messages"][-1].content)


부산의 맑은 하늘 아래, 바닷바람이 살랑이는 모래 위로 나의 첫 발자국이 남았다.


In [12]:
# 9) 서버 종료
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=5)

print("서버 종료 완료")


서버 종료 완료


## 수정 핵심 요약

- `UnsupportedOperation: fileno` 는 노트북 환경의 **stdio subprocess 통신 경로**에서 자주 발생합니다.
- 따라서 Colab/Jupyter에서는 **`stdio` 대신 `http` transport** 를 쓰는 것이 더 안정적입니다.
- LangChain MCP는 `MultiServerMCPClient` 에서 `transport="http"` 와 `url="http://.../mcp"` 설정을 지원합니다.
- FastMCP 서버는 `mcp.run(transport="http", host=..., port=..., path="/mcp")` 로 실행하면 됩니다.
